In [0]:
%run "../config/00_config"

In [0]:
%run "../utils/00_utils"

In [0]:
from pyspark.sql.functions import (
    col,
    count,
    countDistinct,
    when,
    min,
    max,
    sum,
    avg,
    round,
    year,
    month,
    to_timestamp,
    regexp_replace,
    desc
)

from pyspark.sql.types import DecimalType

SOURCE_FILE = "ecommerce_pedidos.csv"
SOURCE_PATH = f"{RAW_BATCH_PATH}{SOURCE_FILE}"

CSV_OPTIONS = {
    "header": "true",
    "inferSchema": "false",
    "sep": ","
}

EXPECTED_COLUMNS = [
    "id_pedido",
    "id_cliente",
    "id_endereco_entrega",
    "dt_pedido",
    "status_pedido",
    "valor_total",
    "valor_frete",
    "metodo_pagamento"
]

adls_options = get_adls_options()

df_raw = read_source_csv(
    spark=spark,
    source_path=SOURCE_PATH,
    adls_options=adls_options,
    csv_options=CSV_OPTIONS
)

actual_columns = df_raw.columns

missing_columns = [c for c in EXPECTED_COLUMNS if c not in actual_columns]
extra_columns = [c for c in actual_columns if c not in EXPECTED_COLUMNS]

if missing_columns:
    raise Exception(f"Colunas obrigatórias ausentes na origem: {missing_columns}")

if extra_columns:
    print(f"Atenção: colunas extras encontradas na origem: {extra_columns}")

print("Leitura da Raw ecommerce_pedidos concluída.")
print(f"Arquivo lido: {SOURCE_PATH}")
print(f"Total de colunas: {len(df_raw.columns)}")

df_raw.printSchema()

display(df_raw.limit(10))

In [0]:
total_linhas = df_raw.count()
ids_distintos = df_raw.select("id_pedido").distinct().count()
ids_duplicados = total_linhas - ids_distintos

print(f"Total de linhas: {total_linhas}")
print(f"IDs de pedido distintos: {ids_distintos}")
print(f"IDs de pedido duplicados: {ids_duplicados}")

display(
    df_raw.select([
        count(
            when(
                col(c).isNull() | (col(c) == ""),
                True
            )
        ).alias(c)
        for c in df_raw.columns
    ])
)

In [0]:
df_pedidos = (
    df_raw
    .withColumn("id_pedido_int", col("id_pedido").cast("int"))
    .withColumn("id_cliente_int", col("id_cliente").cast("int"))
    .withColumn("id_endereco_entrega_int", col("id_endereco_entrega").cast("int"))
    .withColumn("dt_pedido_ts", to_timestamp(col("dt_pedido")))
    .withColumn(
        "valor_total_dec",
        regexp_replace(col("valor_total"), ",", ".").cast(DecimalType(18, 2))
    )
    .withColumn(
        "valor_frete_dec",
        regexp_replace(col("valor_frete"), ",", ".").cast(DecimalType(18, 2))
    )
)

print("Resumo geral da Raw ecommerce_pedidos:")

display(
    df_pedidos.select(
        count("*").alias("total_linhas"),
        countDistinct("id_pedido_int").alias("pedidos_distintos"),
        min("dt_pedido_ts").alias("data_pedido_mais_antiga"),
        max("dt_pedido_ts").alias("data_pedido_mais_recente"),
        round(sum("valor_total_dec"), 2).alias("receita_total_bruta"),
        round(avg("valor_total_dec"), 2).alias("ticket_medio_bruto"),
        round(sum("valor_frete_dec"), 2).alias("frete_total_bruto"),
        round(avg("valor_frete_dec"), 2).alias("frete_medio_bruto")
    )
)

print("Distribuição por status_pedido:")

display(
    df_pedidos
    .groupBy("status_pedido")
    .agg(
        count("*").alias("qtd_pedidos"),
        round(sum("valor_total_dec"), 2).alias("receita_total")
    )
    .orderBy(desc("qtd_pedidos"))
)

print("Quantidade mensal de pedidos:")

display(
    df_pedidos
    .withColumn("ano_pedido", year(col("dt_pedido_ts")))
    .withColumn("mes_pedido", month(col("dt_pedido_ts")))
    .groupBy("ano_pedido", "mes_pedido")
    .agg(
        count("*").alias("qtd_pedidos"),
        round(sum("valor_total_dec"), 2).alias("receita_total"),
        round(avg("valor_total_dec"), 2).alias("ticket_medio")
    )
    .orderBy("ano_pedido", "mes_pedido")
)

print("Pedidos mais antigos:")

display(
    df_pedidos
    .select(
        "id_pedido",
        "id_cliente",
        "id_endereco_entrega",
        "dt_pedido",
        "status_pedido",
        "valor_total",
        "valor_frete",
        "metodo_pagamento"
    )
    .orderBy("dt_pedido_ts")
    .limit(20)
)

print("Pedidos mais recentes:")

display(
    df_pedidos
    .select(
        "id_pedido",
        "id_cliente",
        "id_endereco_entrega",
        "dt_pedido",
        "status_pedido",
        "valor_total",
        "valor_frete",
        "metodo_pagamento"
    )
    .orderBy(desc("dt_pedido_ts"))
    .limit(20)
)